# Optimizer and Learning Rate Scheduler Tutorial

This tutorial covers optimizers and learning rate schedulers available in `kgcnn_torch`:

**Part 1: Optimizer Head-to-Head Comparison** (new)
- Compare Adam, SGD, AdamW, Adamax, Adadelta on the ESOL dataset
- Train the same GCN model with each optimizer and plot loss curves

**Part 2: Learning Rate Schedulers**
- `LinearWarmupScheduler` - warmup then constant
- `LinearWarmupExponentialDecay` - warmup then exponential decay
- `CosineWarmupScheduler` - warmup then cosine annealing
- `PolynomialDecayScheduler` - polynomial decay
- `LinearLearningRateScheduler` - constant then linear decay (Keras-compatible)
- `LinearWarmupLinearLearningRateScheduler` - warmup then linear decay
- `get_scheduler()` factory function
- `ReduceLROnPlateau` from PyTorch
- Visualizing learning rate curves

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

---
## Part 1: Optimizer Head-to-Head Comparison

We train the same GCN model architecture with five different optimizers on the ESOL
solubility dataset (1128 molecules) and compare their training and validation loss curves.
This mirrors the Keras tutorial's optimizer comparison approach.

### Load the ESOL Dataset

In [ ]:
from kgcnn_torch.data.datasets.ESOLDataset import ESOLDataset
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

dataset = ESOLDataset()
print(f"ESOL dataset: {len(dataset)} molecules")

# Train/val split
indices = np.arange(len(dataset))
train_idx, val_idx = train_test_split(indices, test_size=0.25, random_state=42)

train_data = dataset[torch.tensor(train_idx).long()]
val_data = dataset[torch.tensor(val_idx).long()]

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

### Train GCN with Each Optimizer

In [ ]:
from kgcnn_torch.models.gcn import GCNModel
from kgcnn_torch.training.trainer import fit

EPOCHS = 300
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the optimizers to compare
optimizer_configs = {
    "Adam": lambda params: torch.optim.Adam(params, lr=1e-3),
    "SGD": lambda params: torch.optim.SGD(params, lr=1e-2, momentum=0.9),
    "Adamax": lambda params: torch.optim.Adamax(params, lr=1e-3),
    "AdamW": lambda params: torch.optim.AdamW(params, lr=1e-3, weight_decay=0.01),
    "Adadelta": lambda params: torch.optim.Adadelta(params),
}

# Model config (same for all optimizers)
model_config = dict(
    node_dim=64,
    depth=3,
    gcn_units=100,
    gcn_activation="relu",
    node_pooling="sum",
    output_units=[64, 32],
    output_activation="relu",
    output_final_activation="linear",
    num_targets=1,
    output_embedding="graph",
    use_node_embedding=True,
    num_embeddings=95,
)

histories = {}

for opt_name, opt_fn in optimizer_configs.items():
    print(f"\nTraining with {opt_name}...")

    model = GCNModel(**model_config)
    optimizer = opt_fn(model.parameters())

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=nn.L1Loss(),
        epochs=EPOCHS,
        device=device,
        verbose=0,
    )
    histories[opt_name] = history
    print(f"  Final train loss: {history['train_loss'][-1]:.4f}, "
          f"val loss: {history['val_loss'][-1]:.4f}")

print("\nAll optimizers trained.")

### Plot Loss Curves

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 6))

for opt_name, history in histories.items():
    axs[0].plot(history["train_loss"], label=opt_name)
axs[0].set_ylabel("Train Loss (MAE)")
axs[0].set_xlabel("Epochs")
axs[0].set_yscale("log")
axs[0].set_title("Training Loss")
axs[0].legend()
axs[0].grid(True, alpha=0.3)

for opt_name, history in histories.items():
    axs[1].plot(history["val_loss"], label=opt_name)
axs[1].set_ylabel("Validation Loss (MAE)")
axs[1].set_xlabel("Epochs")
axs[1].set_yscale("log")
axs[1].set_title("Validation Loss")
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.suptitle("Optimizer Comparison on ESOL (GCN, 300 epochs)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print(f"{'Optimizer':<12} {'Final Train Loss':>18} {'Final Val Loss':>16} {'Best Val Loss':>15}")
print("-" * 65)
for opt_name, history in histories.items():
    best_val = min(history["val_loss"])
    print(f"{opt_name:<12} {history['train_loss'][-1]:>18.4f} {history['val_loss'][-1]:>16.4f} {best_val:>15.4f}")

---
## Part 2: Learning Rate Schedulers

### PyTorch Optimizers

Standard PyTorch optimizers work directly with `kgcnn_torch` models since all models are native `nn.Module` subclasses.

In [ ]:
# Create a simple model for demonstration
model = nn.Linear(10, 1)

# Common optimizers
opt_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
opt_sgd = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
opt_adamw = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
opt_adamax = torch.optim.Adamax(model.parameters(), lr=1e-3)
opt_rmsprop = torch.optim.RMSprop(model.parameters(), lr=1e-3)

optimizers = {
    "Adam": opt_adam,
    "SGD": opt_sgd,
    "AdamW": opt_adamw,
    "Adamax": opt_adamax,
    "RMSprop": opt_rmsprop,
}

for name, opt in optimizers.items():
    print(f"{name}: lr={opt.defaults['lr']}")

### kgcnn_torch Custom Schedulers

All custom schedulers in `kgcnn_torch` extend `torch.optim.lr_scheduler.LambdaLR`. They implement learning rate patterns commonly used in GNN training. Let us visualize each one.

In [ ]:
from kgcnn_torch.training.scheduler import (
    LinearWarmupScheduler,
    LinearWarmupExponentialDecay,
    CosineWarmupScheduler,
    PolynomialDecayScheduler,
    LinearLearningRateScheduler,
    LinearWarmupLinearLearningRateScheduler,
    get_scheduler,
)

In [ ]:
def collect_lr_curve(scheduler_class, optimizer_cls=torch.optim.Adam, base_lr=1e-3,
                     epochs=300, **kwargs):
    """Simulate a scheduler for `epochs` steps and record the LR."""
    dummy = nn.Linear(1, 1)
    opt = optimizer_cls(dummy.parameters(), lr=base_lr)
    sched = scheduler_class(opt, **kwargs)
    lrs = []
    for _ in range(epochs):
        lrs.append(opt.param_groups[0]['lr'])
        sched.step()
    return lrs

#### LinearWarmupScheduler

Linearly ramps the learning rate from near zero to `base_lr` over `warmup_epochs`, then keeps it constant.

In [ ]:
lrs = collect_lr_curve(LinearWarmupScheduler, warmup_epochs=30, epochs=200)

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("LinearWarmupScheduler (warmup_epochs=30)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### LinearWarmupExponentialDecay

Linear warmup followed by exponential decay: `lr = base_lr * decay_rate^(steps_since_warmup / decay_epochs)`.

In [ ]:
lrs = collect_lr_curve(
    LinearWarmupExponentialDecay,
    warmup_epochs=20, decay_rate=0.96, decay_epochs=10,
    epochs=300
)

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("LinearWarmupExponentialDecay (warmup=20, decay_rate=0.96, decay_epochs=10)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### CosineWarmupScheduler

Linear warmup followed by cosine annealing down to `min_lr_factor * base_lr`.

In [ ]:
lrs = collect_lr_curve(
    CosineWarmupScheduler,
    warmup_epochs=20, total_epochs=300, min_lr_factor=0.01,
    epochs=300
)

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("CosineWarmupScheduler (warmup=20, total=300, min_lr_factor=0.01)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### PolynomialDecayScheduler

Decays from `base_lr` to `lr_final_factor * base_lr` using a polynomial: `lr = (1 - final) * (1 - step/total)^power + final`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))

for i, power in enumerate([0.5, 1.0, 2.0]):
    lrs = collect_lr_curve(
        PolynomialDecayScheduler,
        total_epochs=300, lr_final_factor=0.01, power=power,
        epochs=300
    )
    axes[i].plot(lrs)
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Learning Rate")
    axes[i].set_title(f"PolynomialDecay (power={power})")
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### LinearLearningRateScheduler

Keeps `learning_rate_start` constant until `epo_min`, then linearly decays to `learning_rate_stop` by epoch `epo`. This matches the Keras KGCNN `LinearLearningRateScheduler` behavior.

In [ ]:
lrs = collect_lr_curve(
    LinearLearningRateScheduler,
    learning_rate_start=1e-3, learning_rate_stop=1e-5,
    epo_min=100, epo=500,
    epochs=600
)

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("LinearLearningRateScheduler (start=1e-3, stop=1e-5, epo_min=100, epo=500)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### LinearWarmupLinearLearningRateScheduler

Linear warmup from ~0 to `learning_rate_start` over `epo_warmup`, then linear decay to `learning_rate_stop` by epoch `epo`. Used by MatBench/Materials Project configs.

In [ ]:
lrs = collect_lr_curve(
    LinearWarmupLinearLearningRateScheduler,
    learning_rate_start=1e-3, learning_rate_stop=1e-5,
    epo_warmup=50, epo=500,
    epochs=600
)

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("LinearWarmupLinearLearningRateScheduler (warmup=50, start=1e-3, stop=1e-5, epo=500)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Comparing All Schedulers Side by Side

In [ ]:
epochs = 500
base_lr = 1e-3

schedules = {
    "LinearWarmup": collect_lr_curve(
        LinearWarmupScheduler, base_lr=base_lr, warmup_epochs=30, epochs=epochs),
    "WarmupExponential": collect_lr_curve(
        LinearWarmupExponentialDecay, base_lr=base_lr,
        warmup_epochs=30, decay_rate=0.96, decay_epochs=10, epochs=epochs),
    "CosineWarmup": collect_lr_curve(
        CosineWarmupScheduler, base_lr=base_lr,
        warmup_epochs=30, total_epochs=epochs, min_lr_factor=0.01, epochs=epochs),
    "PolynomialDecay": collect_lr_curve(
        PolynomialDecayScheduler, base_lr=base_lr,
        total_epochs=epochs, lr_final_factor=0.01, power=1.0, epochs=epochs),
    "LinearLR": collect_lr_curve(
        LinearLearningRateScheduler, base_lr=base_lr,
        learning_rate_start=base_lr, learning_rate_stop=1e-5,
        epo_min=50, epo=epochs, epochs=epochs),
    "WarmupLinearLR": collect_lr_curve(
        LinearWarmupLinearLearningRateScheduler, base_lr=base_lr,
        learning_rate_start=base_lr, learning_rate_stop=1e-5,
        epo_warmup=30, epo=epochs, epochs=epochs),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, lrs in schedules.items():
    axes[0].plot(lrs, label=name)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Learning Rate")
axes[0].set_title("All Schedulers (Linear Scale)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

for name, lrs in schedules.items():
    axes[1].plot(lrs, label=name)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("All Schedulers (Log Scale)")
axes[1].set_yscale("log")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### The `get_scheduler()` Factory Function

`get_scheduler()` creates a scheduler by name. This is the preferred way to create schedulers from config files.

In [ ]:
from kgcnn_torch.training.scheduler import get_scheduler

model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Available scheduler names
available_names = [
    "linear_warmup",
    "warmup_exponential",
    "LinearWarmupExponentialDecay",
    "LinearLearningRateScheduler",
    "LinearWarmupLinearLearningRateScheduler",
    "warmup_linear",
    "polynomial_decay",
    "cosine_warmup",
    "step",
    "exponential",
    "cosine",
    "reduce_on_plateau",
    "ReduceLROnPlateau",
]
print("Available scheduler names:")
for n in available_names:
    print(f"  - {n}")

In [ ]:
# Create using get_scheduler
sched1 = get_scheduler("warmup_exponential", optimizer,
                        warmup_epochs=10, decay_rate=0.96, decay_epochs=10)
print("Scheduler 1:", type(sched1).__name__)

# Reset optimizer
optimizer2 = torch.optim.Adam(model.parameters(), lr=1e-3)
sched2 = get_scheduler("cosine_warmup", optimizer2,
                        warmup_epochs=10, total_epochs=200, min_lr_factor=0.01)
print("Scheduler 2:", type(sched2).__name__)

# PyTorch built-in schedulers are also accessible
optimizer3 = torch.optim.Adam(model.parameters(), lr=1e-3)
sched3 = get_scheduler("step", optimizer3, step_size=50, gamma=0.5)
print("Scheduler 3:", type(sched3).__name__)

In [ ]:
# Using get_scheduler from a config dict (typical in training scripts)
scheduler_config = {
    "name": "LinearLearningRateScheduler",
    "config": {
        "learning_rate_start": 1e-3,
        "learning_rate_stop": 1e-5,
        "epo_min": 100,
        "epo": 800
    }
}

opt = torch.optim.Adam(model.parameters(), lr=scheduler_config["config"]["learning_rate_start"])
sched = get_scheduler(scheduler_config["name"], opt, **scheduler_config["config"])
print("Config-created scheduler:", type(sched).__name__)

### ReduceLROnPlateau

`ReduceLROnPlateau` is a PyTorch built-in scheduler that reduces the learning rate when a metric has stopped improving. It is metric-driven, unlike the epoch-based schedulers above. The `fit()` function automatically handles it by passing the validation loss.

In [ ]:
model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Create via get_scheduler
plateau_sched = get_scheduler("reduce_on_plateau", optimizer,
                               mode='min', factor=0.5, patience=10, verbose=False)
print("Type:", type(plateau_sched).__name__)

# Simulate: feed fake loss values
lrs = []
for epoch in range(200):
    lrs.append(optimizer.param_groups[0]['lr'])
    # Simulated loss that plateaus and then improves in steps
    fake_loss = 1.0 / (1 + epoch * 0.01) + 0.05 * np.sin(epoch * 0.1)
    plateau_sched.step(fake_loss)

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("ReduceLROnPlateau (factor=0.5, patience=10)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Combining Scheduler with fit()

The `fit()` function from `kgcnn_torch.training.trainer` handles scheduler stepping automatically:
- For `ReduceLROnPlateau`: calls `scheduler.step(val_loss)` at each epoch
- For all other schedulers: calls `scheduler.step()` at each epoch

In [ ]:
# Example: using fit() with LinearWarmupExponentialDecay scheduler on real ESOL data
from kgcnn_torch.models.gcn import GCNModel
from kgcnn_torch.training.trainer import fit
from torch_geometric.loader import DataLoader

# Reuse real ESOL split from above and keep this section lightweight.
quick_train = [train_data[i] for i in range(min(len(train_data), 256))]
quick_val = [val_data[i] for i in range(min(len(val_data), 128))]

train_loader = DataLoader(quick_train, batch_size=16, shuffle=True)
val_loader = DataLoader(quick_val, batch_size=16)

# Create model, optimizer, scheduler
model = GCNModel(node_dim=32, depth=2, gcn_units=32, output_units=[16],
                 output_final_activation="linear", num_targets=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = get_scheduler("warmup_exponential", optimizer,
                           warmup_epochs=5, decay_rate=0.98, decay_epochs=5)

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=nn.MSELoss(),
    scheduler=scheduler,
    epochs=50,
    verbose=1,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].set_title("Loss Curves")

axes[1].plot(history["lr"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("LR During Training")

plt.tight_layout()
plt.show()


## Summary

**Part 1: Optimizer Comparison**

We compared five optimizers (Adam, SGD, Adamax, AdamW, Adadelta) on the real ESOL dataset
using the same GCN model architecture. Adam and AdamW typically converge fastest for GNN tasks.

**Part 2: Learning Rate Schedulers**

| Scheduler | Description | Key Parameters |
|---|---|---|
| `LinearWarmupScheduler` | Warmup then constant | `warmup_epochs` |
| `LinearWarmupExponentialDecay` | Warmup then exponential decay | `warmup_epochs`, `decay_rate`, `decay_epochs` |
| `CosineWarmupScheduler` | Warmup then cosine annealing | `warmup_epochs`, `total_epochs`, `min_lr_factor` |
| `PolynomialDecayScheduler` | Polynomial decay | `total_epochs`, `lr_final_factor`, `power` |
| `LinearLearningRateScheduler` | Constant then linear decay | `learning_rate_start/stop`, `epo_min`, `epo` |
| `LinearWarmupLinearLearningRateScheduler` | Warmup then linear decay | `learning_rate_start/stop`, `epo_warmup`, `epo` |
| `ReduceLROnPlateau` | Reduce on plateau (metric-driven) | `factor`, `patience`, `mode` |

Use `get_scheduler(name, optimizer, **kwargs)` to create any scheduler by name from a config dictionary.